# Chapter 8: Scanning and Enumeration

> "Measure twice, cut once." Traditional proverb

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Explain host discovery, port scanning, and service enumeration.
2. Distinguish TCP connect, SYN, and UDP scan techniques.
3. Interpret port states and map services to likely vulnerabilities.
4. Explain vulnerability scanning and its limitations.
5. Build and read a simple port-state model in code.

## Key Terms

- **Port**: A numbered endpoint for a network service.
- **Banner grabbing**: Reading service identification strings.
- **Enumeration**: Extracting detailed information from discovered services.
- **CVSS**: Common Vulnerability Scoring System.
- **False positive**: A reported issue that is not actually exploitable.

---

## 8.1 From Discovery to Detail

Once reconnaissance identifies candidate hosts, scanning determines which are alive, which ports are
open, and what services answer. Enumeration then extracts specifics such as software versions, share
names, and user accounts. The progression matters: each step narrows the field and informs the next,
and skipping ahead wastes effort against targets that turn out to be irrelevant.

## 8.2 Port Scanning Techniques

A **TCP connect** scan completes the full three-way handshake and is reliable but noisy. A **SYN** or
half-open scan sends a SYN and infers state from the response without completing the handshake, which is
faster and somewhat stealthier. **UDP** scanning is harder because the protocol is connectionless and
many services do not reply, so absence of a response is ambiguous. Ports are commonly reported as open,
closed, or filtered, where filtered usually indicates a firewall.

## 8.3 Service Enumeration

Open ports are only the beginning. Banner grabbing reads the text a service offers on connection, often
revealing the product and version. Protocol-specific enumeration goes further: querying SMB for shares,
SNMP for device details, or a web server for technologies. Accurate version information is what lets a
tester move from a list of open ports to a concrete hypothesis about exploitable weaknesses.

## 8.4 Vulnerability Scanning

A vulnerability scanner automates the comparison of discovered software against databases of known
issues, often referencing CVE identifiers {cite}`cve_mitre`. Scanners are efficient but imperfect: they
produce false positives that must be validated and false negatives for issues they do not test. A scan
result is a starting point for analysis, not a verdict.

## 8.5 Why This Matters

Defenders run the same scans against their own networks to find exposures before attackers do.
Understanding scan behavior also helps blue teams tune detection so that hostile scanning stands out
from legitimate administrative activity.

## 8.6 News in Focus

Internet-wide scanning services now continuously index exposed devices and services across the public
internet, making it trivial to locate, for example, databases left open without authentication. Their
existence has repeatedly surfaced large numbers of misconfigured systems and reinforces that exposure is
discovered in minutes, not months.

## 8.7 Worked Example: Modeling Port States

The code models scan results and summarizes open services and a naive risk weighting, illustrating how
raw scan data becomes a prioritized view.


In [1]:
scan = {
    22:  ("open",     "OpenSSH 8.2"),
    80:  ("open",     "nginx 1.18"),
    443: ("open",     "nginx 1.18"),
    3306:("open",     "MySQL 5.7"),
    23:  ("filtered", "telnet"),
    21:  ("open",     "vsftpd 2.3.4"),
}

weights = {"telnet": 5, "ftp": 4, "mysql": 4, "ssh": 1, "nginx": 1}

print(f"{'Port':<6}{'State':<10}{'Service':<18}{'Weight'}")
print("-" * 42)
total = 0
for port, (state, svc) in sorted(scan.items()):
    key = next((k for k in weights if k in svc.lower() or k in ({21:'ftp',3306:'mysql'}.get(port,''))), None)
    w = weights.get(key, 2) if state == "open" else 0
    total += w
    print(f"{port:<6}{state:<10}{svc:<18}{w}")

print(f"\nAggregate exposure weight (open services): {total}")
print("Priority: review the highest-weight legacy services (telnet, ftp) first.")


Port  State     Service           Weight
------------------------------------------
21    open      vsftpd 2.3.4      4
22    open      OpenSSH 8.2       1
23    filtered  telnet            0
80    open      nginx 1.18        1
443   open      nginx 1.18        1
3306  open      MySQL 5.7         4

Aggregate exposure weight (open services): 11
Priority: review the highest-weight legacy services (telnet, ftp) first.


## 8.8 Review Questions (MCQ)

**Q1.** A port reported as filtered most commonly indicates:
A. No service  B. A firewall  C. A crashed host  D. An open service

**Q2.** Which scan completes the full handshake?
A. SYN scan  B. TCP connect scan  C. UDP scan  D. FIN scan

**Q3.** A scanner reporting an issue that is not actually exploitable is a:
A. False negative  B. True positive  C. False positive  D. Zero day

*Answers: Q1 B, Q2 B, Q3 C.*

## 8.9 Lab Assignment

In an isolated lab, scan a deliberately vulnerable virtual machine with a port scanner. Record open
ports, grab banners where possible, and produce a short table prioritizing services for further
analysis. Do not scan systems you do not own or lack authorization to test.

## References

```{bibliography}
:filter: docname in docnames
```
